# Ground-truth parameter selection

This notebook confirms how `OutputConfig.selected_indices` chooses which Gaussian-mixture
parameter maps (amplitude, mean, sigma per component) become target channels, and how
`PatchDataset._build_output_tensor` extracts exactly those channels from the full parameter
stack.

Modules exercised:

- `configuration.dataset_config.OutputConfig`
- `pipelines.dataset_pipeline.datasets.PatchDataset`

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy             as np
import matplotlib.pyplot as plt

import matplotlib
matplotlib.rcParams["figure.dpi"] = 110
matplotlib.rcParams["image.interpolation"] = "nearest"

SEED = 0
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
except Exception:
    pass

print("repo root on path:", REPO_ROOT)



## Synthetic parameter stack

A two-component mixture has six parameter maps ordered as
[a1, mu1, sig1, a2, mu2, sig2]. We give each map a recognisable constant offset so selection
is visible.


In [ ]:
from tools.monitoring.logger                        import Logger
from configuration.dataset_config        import InputConfig, OutputConfig
from tools.data.representation        import Representation
from pipelines.dataset_pipeline.spatial  import Patcher
from pipelines.dataset_pipeline.datasets import PatchDataset

logger      = Logger(log_dir="/tmp/dataset_nb_logs", name="nb05", level="WARNING")
n_gaussians = 2
H, W        = 24, 24

role_labels = []
for g in range(n_gaussians):
    role_labels += [f"a{g+1}", f"mu{g+1}", f"sig{g+1}"]

base   = np.arange(n_gaussians * 3, dtype=np.float32)
params = base[:, None, None] + 0.1 * np.random.default_rng(SEED).random((n_gaussians * 3, H, W)).astype(np.float32)
print("full parameter stack:", params.shape, " roles:", role_labels)



## Index selection for different role subsets

We compute selected indices for the full set and for amplitude-only and mean+sigma subsets.


In [ ]:
full      = OutputConfig(use_amplitude=True,  use_mu=True,  use_sigma=True)
amp_only  = OutputConfig(use_amplitude=True,  use_mu=False, use_sigma=False)
mu_sigma  = OutputConfig(use_amplitude=False, use_mu=True,  use_sigma=True)

for name, cfg in [("full", full), ("amp_only", amp_only), ("mu_sigma", mu_sigma)]:
    idx = cfg.selected_indices(n_gaussians)
    print(f"{name:9s}: roles={cfg.role_names}  indices={idx}  channels={[role_labels[i] for i in idx]}")



## Build output tensors and compare

Each configuration produces an output tensor whose channels are exactly the selected parameter
maps. We display the full stack and the amplitude-only selection.


In [ ]:
stack       = np.zeros((1, H, W), dtype=np.complex64)
input_cfg   = InputConfig(use_primary=True, primary_representation=Representation.MAG_ONLY,
                          use_secondaries=False, use_interferograms=False, use_dem=False)
patcher     = Patcher.build(spatial_size=(H, W), patch_size=(H, W), stride=H)

def output_for(cfg):
    ds = PatchDataset(inputs=stack, gt_parameters=params, grid=patcher, input_config=input_cfg,
                      output_config=cfg, split_name="val", n_secondaries=0, n_interferograms=0,
                      n_gaussians=n_gaussians)
    _, gt = ds[0]
    return gt

gt_full = output_for(full)
gt_amp  = output_for(amp_only)
print("full output channels:", gt_full.shape[0], " amp-only channels:", gt_amp.shape[0])

idx_full = full.selected_indices(n_gaussians)
fig, axes = plt.subplots(1, gt_full.shape[0], figsize=(1.8 * gt_full.shape[0], 2.2), squeeze=False)
for c in range(gt_full.shape[0]):
    ax = axes[0, c]
    im = ax.imshow(gt_full[c], cmap="cividis")
    ax.set_title(role_labels[idx_full[c]], fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()


In [ ]:
idx_amp = amp_only.selected_indices(n_gaussians)
err = float(np.abs(gt_amp - params[idx_amp]).max())
print("amplitude-only selection matches source maps, max error:", f"{err:.3e}")
print("selected amplitude roles:", [role_labels[i] for i in idx_amp])


## Expected visual outcome

The full output tensor should show six panels in the order a1, mu1, sig1, a2, mu2, sig2, each
displaying the constant offset of its source map. The amplitude-only configuration should
select exactly channels 0 and 3 with zero extraction error. Index lists should follow the
pattern base = g*3 plus the per-role offset.